# 07 · 100 Hz CNN, timing pilot and frozen searches

Preparation milestone of Plan 003: restore input fidelity, measure development-only compute, and freeze the proposed searches **before outer evaluation**. This notebook currently implements `pilot` and `freeze`; full nested evaluation is the next milestone, not a result claimed here. [Preparation report](../docs/portfolio-tuning-preparation.md).

Use the shared `APNEA_PORTFOLIO_RUN` from Notebooks 05–06. `APNEA_PORTFOLIO_STAGE=pilot` is the default. `freeze` requires a completed matching pilot. Synthetic smoke runs are marked and cannot freeze a real protocol. All data remain retrospective and exploratory, with original patient identities unverified.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import pickle
import platform
import random
import resource
import subprocess
import sys
import tempfile
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid, ParameterSampler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def root_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


RUN = root_path(
    os.environ.get("APNEA_PORTFOLIO_RUN", "data/processed/portfolio-v2/plan001-real-20261108")
)
PHASE1 = root_path(os.environ.get("APNEA_RUN", "data/processed/phase1"))
FEATURES = root_path(os.environ.get("APNEA_PHASE2_RUN", "data/processed/phase2-grouped"))
STAGE = os.environ.get("APNEA_PORTFOLIO_STAGE", "pilot")
SMOKE = os.environ.get("APNEA_PORTFOLIO_SMOKE") == "1"
torch.set_num_threads(4)

## 1. Fixed architecture and deterministic candidate lists

The original convolution widths, kernels, strides, pooling and linear head remain identical: 3,009 parameters. All inputs have 6,000 filtered samples at 100 Hz. Standardize inside the minute with a .001 floor, clip to ±8, and retain every eligible window. Existing 25 Hz checkpoints are incompatible.

Each model gets 24 candidates, shared across outer folds. LR and the adapted linear SVM use explicit grids; RF/XGBoost use seeded random samples. The CNN varies training settings only. No architecture search or probability calibration is introduced.

In [ ]:
def make_cnn() -> nn.Module:
    return nn.Sequential(
        nn.Conv1d(1, 8, 15, stride=2, padding=7),
        nn.ReLU(),
        nn.MaxPool1d(2),
        nn.Conv1d(8, 16, 7, stride=2, padding=3),
        nn.ReLU(),
        nn.MaxPool1d(2),
        nn.Conv1d(16, 24, 5, stride=2, padding=2),
        nn.ReLU(),
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(24, 1),
    )


def normalize_minute(minute, fs=100):
    minute = np.asarray(minute, dtype=float)
    if fs != 100 or minute.shape != (6000,) or not np.isfinite(minute).all():
        raise ValueError("Expected finite 6000-sample input at 100 Hz")
    return np.clip((minute - minute.mean()) / max(float(minute.std()), 0.001), -8, 8).astype(
        "float32"
    )


def load_waveforms(cohort, phase1):
    cohort = cohort.reset_index(drop=True)
    result = np.empty((len(cohort), 6000), dtype="float32")
    for record, part in cohort.groupby("record_id"):
        with np.load(Path(phase1) / "waveforms" / f"{record}.npz") as source:
            signal = source["filtered"]
            fs = int(source["fs"])
            for row in part.itertuples():
                result[row.Index] = normalize_minute(signal[row.start_sample : row.end_sample], fs)
    return result


def candidate_registry():
    result = {
        "logistic_regression": list(
            ParameterGrid(
                dict(
                    C=[0.0001, 0.001, 0.01, 0.1, 0.3, 1, 3, 10, 30, 100, 1000, 10000],
                    class_weight=[None, "balanced"],
                )
            )
        ),
        "linear_svm": list(
            ParameterGrid(
                dict(
                    C=[0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 1000],
                    class_weight=[None, "balanced"],
                )
            )
        ),
    }
    result["random_forest"] = list(
        ParameterSampler(
            dict(
                n_estimators=[200, 400, 800],
                max_depth=[None, 4, 8, 12, 20],
                min_samples_leaf=[1, 2, 5, 10, 20],
                min_samples_split=[2, 5, 10],
                max_features=[1.0, "sqrt", 0.5],
                class_weight=[None, "balanced"],
            ),
            n_iter=24,
            random_state=1067,
        )
    )
    result["xgboost"] = list(
        ParameterSampler(
            dict(
                n_estimators=[100, 200, 400, 800],
                max_depth=[2, 3, 4, 6],
                learning_rate=loguniform(0.01, 0.2),
                min_child_weight=[1, 3, 5, 10],
                subsample=[0.6, 0.8, 1],
                colsample_bytree=[0.6, 0.8, 1],
                reg_alpha=[0, 0.01, 0.1, 1],
                reg_lambda=[0.1, 1, 10],
            ),
            n_iter=24,
            random_state=1067,
        )
    )
    result["cnn"] = [
        dict(learning_rate=0.001, weight_decay=0.0001, batch_size=128, pos_weight="balanced")
    ] + list(
        ParameterSampler(
            dict(
                learning_rate=loguniform(0.0001, 0.003),
                weight_decay=loguniform(0.000001, 0.01),
                batch_size=[32, 64, 128],
                pos_weight=[1, "balanced"],
            ),
            n_iter=23,
            random_state=1067,
        )
    )
    return json.loads(json.dumps(result, default=lambda x: x.item()))


def make_classical_model(name, settings, seed=67):
    if name == "logistic_regression":
        estimator = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=seed, **settings)
    elif name == "linear_svm":
        estimator = SVC(kernel="linear", probability=False, random_state=seed, **settings)
    elif name == "random_forest":
        estimator = RandomForestClassifier(n_jobs=4, random_state=seed, **settings)
    elif name == "xgboost":
        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=1,
            n_jobs=4,
            random_state=seed,
            **settings,
        )
    else:
        raise ValueError("Unknown model")
    steps = [SimpleImputer(strategy="median")]
    if name in ["logistic_regression", "linear_svm"]:
        steps.append(StandardScaler())
    return make_pipeline(*steps, estimator)


def benchmark_xgboost(x, y, settings, seed=67):
    # Native torch/XGBoost interaction is reproducibly unsafe in this environment.
    worker = """import json, pickle, resource, sys, time
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
x,y,settings,seed=pickle.loads(sys.stdin.buffer.read())
model=make_pipeline(
    SimpleImputer(strategy="median"),
    XGBClassifier(objective="binary:logistic",eval_metric="logloss",
                  tree_method="hist",scale_pos_weight=1,n_jobs=4,
                  random_state=seed,**settings))
start=time.perf_counter();model.fit(x,y)
print(json.dumps(dict(fit_seconds=time.perf_counter()-start,
    torch_loaded="torch" in sys.modules,
    peak_rss=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)))
"""
    start = perf_counter()
    completed = subprocess.run(
        [sys.executable, "-c", worker],
        input=pickle.dumps((np.asarray(x), np.asarray(y), settings, seed)),
        capture_output=True,
        check=True,
        timeout=300,
    )
    result = json.loads(completed.stdout)
    if result["torch_loaded"]:
        raise ValueError("XGBoost worker unexpectedly loaded torch")
    result["process_seconds"] = perf_counter() - start
    return result


def reset_seed(seed=67):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    count = 0
    for x, y in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x).squeeze(1), y)
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite training loss")
        loss.backward()
        if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
            raise ValueError("Missing/nonfinite gradients")
        optimizer.step()
        total += float(loss.detach()) * len(y)
        count += len(y)
    return total / count


def write_stage(run, stage, payload):
    target = Path(run) / stage
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    if target.exists():
        try:
            manifest = json.loads((target / "manifest.json").read_text())
            actual = hashlib.sha256((target / "payload.json").read_bytes()).hexdigest()
            if manifest["identity_hash"] != digest or actual != manifest["payload_hash"]:
                raise ValueError("Stage content mismatch")
        except (KeyError, FileNotFoundError, json.JSONDecodeError) as exc:
            raise ValueError("Corrupt/incomplete stage") from exc
        return
    Path(run).mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=f".{stage}-", dir=run) as tmp:
        directory = Path(tmp) / stage
        directory.mkdir()
        content = json.dumps(payload, indent=2, sort_keys=True) + "\n"
        (directory / "payload.json").write_text(content)
        (directory / "manifest.json").write_text(
            json.dumps(
                dict(
                    identity_hash=digest, payload_hash=hashlib.sha256(content.encode()).hexdigest()
                ),
                indent=2,
            )
            + "\n"
        )
        directory.rename(target)


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def verified_payload(run, stage):
    target = Path(run) / stage
    payload = json.loads((target / "payload.json").read_text())
    write_stage(run, stage, payload)
    return payload


def verify_dependency(path):
    manifest = json.loads((path / "manifest.json").read_text())
    for name, digest in manifest["artifacts"].items():
        if hash_file(path / name) != digest:
            raise ValueError("Dependency artifact checksum mismatch")
    return manifest

## 2. Verify dependencies and freeze all scientific choices

The source and artifact identities, exact candidate lists, folds, seeds, metrics, epoch-selection rules, ablation settings and hardware limits travel with the freeze. A freeze is a preparation artifact, not fitted weights or performance evidence. Existing protocol files remain immutable; the freeze links their identities rather than rewriting their draft manifests.

In [ ]:
if STAGE not in {"pilot", "freeze"}:
    raise ValueError("Only pilot/freeze are implemented in this preparation milestone")
protocol = verify_dependency(RUN / "protocol")
baselines = verify_dependency(RUN / "baselines")
if protocol["configuration"]["synthetic"] != SMOKE or baselines["sources"]["synthetic"] != SMOKE:
    raise ValueError("Smoke/real mismatch")
if baselines["sources"]["protocol_identity"] != protocol["identity_hash"]:
    raise ValueError("Baseline/protocol mismatch")
if (
    not baselines["specification"]["main_matrix"]
    or baselines["specification"]["status"] != "adapted"
):
    raise ValueError("Expected the verified adapted literature method")
input_hashes = {}
if not SMOKE:
    phase1_manifest = json.loads((PHASE1 / "manifest.json").read_text())
    for name, digest in phase1_manifest["artifact_sha256"].items():
        if name.startswith("waveforms/"):
            if hash_file(PHASE1 / name) != digest:
                raise ValueError("Source waveform checksum mismatch")
            input_hashes[name] = digest
    input_hashes["features"] = hash_file(FEATURES / "features.parquet")
    if input_hashes["features"] != protocol["sources"]["features"]:
        raise ValueError("Feature source differs from the saved protocol")

cohort = pd.read_parquet(RUN / "protocol/cohort.parquet")
folds = pd.read_csv(RUN / "protocol/fold_assignments.csv")
stops = pd.read_csv(RUN / "protocol/cnn_stop_assignments.csv")
registry = candidate_registry()
if SMOKE:
    registry = {name: values[:2] for name, values in registry.items()}
identity = dict(
    input_hashes=input_hashes,
    synthetic=SMOKE,
    protocol_identity=protocol["identity_hash"],
    baseline_identity=baselines["identity_hash"],
    candidates=registry,
    architecture=str(make_cnn()),
    parameters=sum(p.numel() for p in make_cnn().parameters()),
    input_hz=100,
    input_samples=6000,
    device="cpu",
    xgboost_execution="fresh subprocess without torch; same estimator/data/search",
    threads=4,
    source_hashes={
        name: hash_file(ROOT / "notebooks" / name)
        for name in [
            "03_waveform_cnn.ipynb",
            "05_evaluation_protocol.ipynb",
            "06_literature_baselines.ipynb",
            "07_nested_model_comparison.ipynb",
        ]
    },
    lock_sha256=hash_file(ROOT / "uv.lock"),
    versions={
        p: importlib.metadata.version(p)
        for p in ["numpy", "scipy", "pandas", "scikit-learn", "torch", "xgboost"]
    },
    platform=platform.platform(),
    python=platform.python_version(),
    rules=dict(
        primary_seed=67,
        repeat_seeds=[167, 267],
        candidate_seed=1067,
        fit_seed="training_seed + 10000 * outer_fold; fixed across candidates/inner stages",
        candidate_selection="mean inner-fold AP; ties within 1e-12 choose earlier candidate",
        threshold="maximize selected inner OOF F1; lowest threshold on ties",
        cnn_max_epochs=2 if SMOKE else 60,
        cnn_patience=8,
        cnn_min_delta=0.0001,
        cnn_checkpoint=(
            "earliest improving unweighted stopping BCE; training pos_wei"
            "ght from current fitting labels only"
        ),
        cnn_refit=(
            "fresh inner model on all inner-training groups for selected "
            "epochs; no inner-validation stopping"
        ),
        cnn_outer_epochs="round-half-up median selected inner calibrated epochs, clipped [1,60]",
        cnn_normalization="per-minute zscore with .001 floor; clip [-8,8]; no resampling",
        sklearn_search=(
            "explicit saved inner indices; AP; refit=False; n_jobs=1; error_score=raise"
        ),
        preprocessing=(
            "fit-only imputation/scaling/class weights; labels/groups/IDs excluded as predictors"
        ),
        bootstrap=(
            "paired subject-group multiplicities; seed 20261108; 2000 dra"
            "ws; conditional fixed predictions"
        ),
        primary_metric="arithmetic mean outer-fold average_precision_score; pooled OOF secondary",
        rq1_fixed={
            "logistic_regression": dict(C=1, max_iter=1000),
            "random_forest": dict(n_estimators=200, max_depth=8, min_samples_leaf=5),
            "xgboost": dict(
                n_estimators=200, max_depth=3, learning_rate=0.05, subsample=1, colsample_bytree=1
            ),
        },
        inference_scope=(
            "exploratory internal retrospective; identities unverified; prior test exposure"
        ),
    ),
)
print(
    "Stage:",
    STAGE,
    "Synthetic:",
    SMOKE,
    "Candidate counts:",
    {k: len(v) for k, v in registry.items()},
)

## 3. Development-only timing pilot

Only outer-fold-0 training records excluding inner-fold-0 validation may enter the pilot. Sample labels for the tiny learning diagnostic come from that fitting set. No outer scores, hyperparameter ranking or threshold selection occur. Measure two full epochs per batch size (one in synthetic smoke), representative classical fits, and peak process RSS. Input loading is timed separately. Extrapolations are planning scenarios, not runtime guarantees.

In [ ]:
if STAGE == "pilot":
    if (RUN / "pilot").exists():
        pilot = verified_payload(RUN, "pilot")
        if pilot["identity"] != identity:
            raise ValueError("Pilot dependency/configuration drift; use a new run")
    else:
        fit_records = folds.query(
            "outer_fold == 0 and outer_role == 'train' and inner_fold != 0"
        ).record_id
        training = cohort[cohort.record_id.isin(fit_records)].reset_index(drop=True)
        assert set(training.label) == {0, 1}
        assert not set(training.record_id) & set(
            folds.query("outer_fold == 0 and outer_role == 'test'").record_id
        )
        loading_start = perf_counter()
        if SMOKE:
            rng = np.random.default_rng(67)
            waveforms = rng.normal(size=(len(training), 6000)).astype("float32")
            f2 = pd.DataFrame(rng.normal(size=(len(training), 6)))
        else:
            waveforms = load_waveforms(training, PHASE1)
            raw = pd.read_parquet(FEATURES / "features.parquet")
            names = [
                "mean_rr_ms",
                "median_rr_ms",
                "mean_hr_bpm",
                "sdrr_ms",
                "rmssd_ms",
                "prr50_pct",
            ]
            merged = training[["record_id", "minute", "label"]].merge(
                raw[["record_id", "minute", "label", *names]],
                on=["record_id", "minute"],
                validate="one_to_one",
                suffixes=("", "_source"),
            )
            assert merged.label.equals(merged.label_source)
            f2 = merged[names]
        literature = pd.read_parquet(RUN / "baselines/features.parquet")
        literature = training[["record_id", "minute"]].merge(
            literature, on=["record_id", "minute"], validate="one_to_one"
        )
        linear_names = baselines["specification"]["feature_order"]
        loading_seconds = perf_counter() - loading_start
        y = training.label.to_numpy()
        x = torch.from_numpy(waveforms[:, None, :])
        target = torch.tensor(y, dtype=torch.float32)
        timings = []
        references = {
            "logistic_regression": dict(C=1, class_weight=None),
            "linear_svm": dict(C=1, class_weight=None),
            "random_forest": dict(n_estimators=200, max_depth=8, min_samples_leaf=5),
            "xgboost": dict(n_estimators=200, max_depth=3, learning_rate=0.05),
        }
        for name, settings in references.items():
            reset_seed()
            values = literature[linear_names] if name == "linear_svm" else f2
            if name == "xgboost":
                result = benchmark_xgboost(values, y, settings)
                seconds = result["fit_seconds"]
                print("XGBoost isolated fit seconds:", seconds, flush=True)
            else:
                model = make_classical_model(name, settings)
                start = perf_counter()
                model.fit(values, y)
                seconds = perf_counter() - start
            timings.append(
                dict(model=name, seconds=seconds, n_windows=len(training), settings=settings)
            )
        epochs = 1 if SMOKE else 2
        ratio = float(np.sum(y == 0) / np.sum(y == 1))
        for batch in [32, 64, 128]:
            reset_seed()
            model = make_cnn()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
            criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(ratio))
            loader = DataLoader(
                TensorDataset(x, target), batch_size=batch, shuffle=True, num_workers=0
            )
            history = []
            for epoch in range(epochs):
                start = perf_counter()
                loss = train_epoch(model, loader, optimizer, criterion)
                history.append(dict(epoch=epoch + 1, loss=loss, seconds=perf_counter() - start))
            timings.append(
                dict(
                    model="cnn",
                    batch_size=batch,
                    n_windows=len(training),
                    epochs=history,
                    seconds_per_epoch=float(np.mean([r["seconds"] for r in history])),
                )
            )
            print("Batch", batch, "seconds/epoch", timings[-1]["seconds_per_epoch"], flush=True)
        # Diagnostic only: an intentionally tiny balanced subset of fitting data.
        tiny = np.concatenate([np.flatnonzero(y == label)[:8] for label in [0, 1]])
        reset_seed()
        model = make_cnn()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
        criterion = nn.BCEWithLogitsLoss()
        trace = []
        loader = DataLoader(
            TensorDataset(x[tiny], target[tiny]), batch_size=len(tiny), shuffle=False
        )
        for step in range(4 if SMOKE else 100):
            loss = train_epoch(model, loader, optimizer, criterion)
            if step % 10 == 0 or step == 99:
                trace.append(dict(step=step + 1, loss=loss))
        if not np.isfinite(loss) or (not SMOKE and loss >= trace[0]["loss"]):
            raise ValueError("Tiny fitting sanity check failed; investigate before freezing")
        seconds_per_window = np.mean(
            [t["seconds_per_epoch"] / t["n_windows"] for t in timings if t["model"] == "cnn"]
        )
        # Size-scaled training workload, including calibration and repeat seeds.
        train_window_epochs = 0
        for outer in sorted(folds.outer_fold.unique()):
            outer_train = cohort[
                cohort.record_id.isin(
                    folds.query(
                        "outer_fold == @outer and outer_role == 'train'",
                        local_dict={"outer": outer},
                    ).record_id
                )
            ]
            for inner in sorted(
                folds.query(
                    "outer_fold == @outer and outer_role == 'train'", local_dict={"outer": outer}
                ).inner_fold.unique()
            ):
                inner_n = len(
                    outer_train[
                        outer_train.record_id.isin(
                            folds.query(
                                (
                                    "outer_fold == @outer and outer_role == 'train' "
                                    "and inner_fold != @inner"
                                ),
                                local_dict={"outer": outer, "inner": inner},
                            ).record_id
                        )
                    ]
                )
                calibration_n = len(
                    cohort[
                        cohort.record_id.isin(
                            stops.query(
                                (
                                    "outer_fold == @outer and inner_fold == @inner and role == 'c"
                                    "alibration-fit'"
                                ),
                                local_dict={"outer": outer, "inner": inner},
                            ).record_id
                        )
                    ]
                )
                train_window_epochs += (
                    len(registry["cnn"]) * (inner_n + calibration_n) + 2 * inner_n
                )
            train_window_epochs += 3 * len(outer_train)
        scenarios = {
            str(e): seconds_per_window * train_window_epochs * e / 3600 for e in [10, 30, 60]
        }
        pilot = dict(
            identity=identity,
            complete=True,
            training_windows=len(training),
            training_records=sorted(training.record_id.unique()),
            loading_seconds=loading_seconds,
            timings=timings,
            tiny_fit_trace=trace,
            peak_process_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            * (1 if platform.system() == "Darwin" else 1024),
            input_cache_bytes=len(cohort) * 6000 * 4,
            cnn_training_hours_scenarios=scenarios,
            estimate_limitations=(
                "training-only size extrapolation; excludes stopping validati"
                "on, IO, scheduling, grid extremes and failures; add headroom"
                "; not a runtime guarantee"
            ),
            fit_counts=dict(
                core_classical=1140, linear_svm=380, cnn=725, rq1=300, repeat_seeds=120
            ),
        )
        write_stage(RUN, "pilot", pilot)
    display(pd.DataFrame([{k: v for k, v in t.items() if k != "epochs"} for t in pilot["timings"]]))
    print("CNN training hours at 10/30/60 epochs:", pilot["cnn_training_hours_scenarios"])
    print("Tiny-fit trace:", pilot["tiny_fit_trace"])
    print("Peak process RSS bytes:", pilot["peak_process_rss_bytes"])
else:
    pilot = verified_payload(RUN, "pilot")
    if not pilot.get("complete") or pilot["identity"] != identity:
        raise ValueError("A matching completed timing pilot is required")
    freeze = dict(
        status="frozen_synthetic" if SMOKE else "frozen",
        evaluation_status="not_run",
        identity=identity,
        timing=pilot,
        methods=[r for r in baselines["register"] if r["main_matrix"]],
        scope=("Search definitions and evaluation choices only; no outer performance generated"),
        next_milestone=(
            "implement/verify full nested search runner; retain frozen choices and dependencies"
        ),
    )
    write_stage(RUN, "freeze", freeze)
    print("Frozen preparation:", RUN / "freeze/payload.json")

## Interpretation

Keep the measured cost, candidate lists and freeze manifest with the study. No outer predictions exist from this preparation notebook. A future full-search runner must verify this freeze and implement the saved grouped calibration/refit rules; it must not alter candidates based on outer scores. Report any scientifically necessary amendment as a new version before evaluation. Full nested comparison, repeat seeds, RQ1 reruns and the final portfolio result report remain the next Plan 003 milestone.